In [11]:
%load_ext autoreload
%autoreload 2

Import packages and set up the environment

In [33]:
import os
from dotenv import load_dotenv
from scraper import Website
from IPython.display import Markdown, display
from openai import OpenAI
from openai.types.responses import Response

In [2]:
# Load environment variables
load_dotenv(override=True) #? override=True means that if the environment variable is already set, it will be overridden.
api_key = os.getenv("OPENAI_API_KEY")

# check the key

if not api_key:
    print("OPENAI_API_KEY is not set")
    raise ValueError("OPENAI_API_KEY is not set")
elif not api_key.startswith("sk-proj-"):
    print("OPENAI_API_KEY is not a valid API key")
    raise ValueError("OPENAI_API_KEY is not a valid API key")
elif api_key.strip() != api_key:
    print("OPENAI_API_KEY contains whitespace")
    raise ValueError("OPENAI_API_KEY contains whitespace")
else:
    print("OPENAI_API_KEY is set and valid")

OPENAI_API_KEY is set and valid


Define the message and the messages list

In [3]:
message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [
    {"role": "user", "content": message}
]

Set up the OpenAI client

In [4]:
openai = OpenAI()

response = openai.responses.create(model="gpt-6-luna", reasoning={"effort": "none"}, input=messages)

response.output_text

'Hi! Welcome—it’s lovely to meet you. What would you like to talk about?'

Fetch a website

In [19]:
mamad = Website("https://mamadfar.dev")
print(mamad.contents())

Lo-fi Harbor — play a rainy town that is a CV

Mohammad Farhadi
LO-FI HARBOR
17:43
Thu, Nov 19
Ask MØ
Privacy
CV
?
Midnight Compile
🎵 jukebox ·
paused
RAIN
34%
🌦
🔊
☀️
🍂
❄️
🌸
🔄
☾
NIGHT
🏘️ Harbor
🏎️ Racer
🎯 Range
🤖 Flight
🧠 Pairs
MOHAMMAD FARHADI
Welcome to Lo-fi Harbor
A tiny rainy town where my whole career lives. Chat with my robot MØ, drag the sun into the sea to make it night, paint stars, fish at the pier, hunt vinyls — or just crank the rain and vibe. 🌧️
▶ Enter the Harbor
headphones on · sound & rain included
Lo-fi Harbor — Mohammad Farhadi, Senior Software Engineer
Lo-fi Harbor — play a rainy town that is a CV


In [30]:
system_prompt = """
You are an assistant that analyzes the contents of a website,
and provides a short summary (less than 2000 characters), ignoring text that might be navigation related.
Also, make sure the text is concise, English and have bullet points if there are multiple items.
Responsd in markdown format.Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [11]:
user_prompt_prefix = """
Here are the contents of a website:
Provide a short summary of this website in markdown.
If it includes news or announcements, then summarize these too.
"""

In [20]:
def messages_for(website: Website) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website.contents()}
    ]

In [21]:
messages_for(mamad)

[{'role': 'system',
  'content': '\nYou are an assistant that analyzes the contents of a website,\nand provides a short summary, ignoring text that might be navigation related.\nResponsd in markdown format.Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website:\nProvide a short summary of this website in markdown.\nIf it includes news or announcements, then summarize these too.\nLo-fi Harbor — play a rainy town that is a CV\n\nMohammad Farhadi\nLO-FI HARBOR\n17:43\nThu, Nov 19\nAsk MØ\nPrivacy\nCV\n?\nMidnight Compile\n🎵 jukebox ·\npaused\nRAIN\n34%\n🌦\n🔊\n☀️\n🍂\n❄️\n🌸\n🔄\n☾\nNIGHT\n🏘️ Harbor\n🏎️ Racer\n🎯 Range\n🤖 Flight\n🧠 Pairs\nMOHAMMAD FARHADI\nWelcome to Lo-fi Harbor\nA tiny rainy town where my whole career lives. Chat with my robot MØ, drag the sun into the sea to make it night, paint stars, fish at the pier, hunt vinyls — or just crank the rain and vibe. 🌧️\n▶ Enter the Harbor\nheadphon

In [22]:
def summarize(website: Website) -> str:
    response = openai.responses.create(model="gpt-6-luna", reasoning={"effort": "none"}, input=messages_for(website))
    return response.output_text

In [ ]:
summarize(mamad)

'### خلاصه\n\nرهیاب مهاجر یک پلتفرم برای ایرانیان مهاجر در سراسر جهان است که کسب\u200cوکارها و خدمات ایرانی را معرفی می\u200cکند و به کاربران کمک می\u200cکند شغل، تخفیف، رویداد، مسکن و نیازمندی\u200cهای مرتبط با زندگی در مهاجرت را پیدا کنند. دسته\u200cبندی\u200cهای خدمات آن شامل حقوقی و مهاجرت، پزشکی، رستوران، آموزش، املاک و موارد دیگر است.\n\n**اعلان:** برخی بخش\u200cها، از جمله املاک و بلاگ و اخبار، در سایت با عنوان «به\u200cزودی» معرفی شده\u200cاند.'

In [23]:
def display_summary(website: Website) -> None:
    summary = summarize(website)
    display(Markdown(summary))

In [24]:
display_summary(mamad)

## Lo-fi Harbor

Lo-fi Harbor is Mohammad Farhadi’s interactive CV, presented as a cozy, rainy town. Visitors can explore activities, chat with the robot MØ, and adjust the weather and atmosphere while learning about his career. The site also features a lo-fi jukebox.

In [31]:
cnn = Website("https://cnn.com")
display_summary(cnn)

- CNN’s website provides breaking news, reporting, and videos across topics including US and world news, politics, business, health, entertainment, sports, science, climate, and weather.
- The page also highlights ongoing coverage of the Ukraine-Russia war and Israel-Hamas war.
- No specific news stories or announcements are included in the supplied content.

In [34]:

system_prompt = """
tell me about the website and the contents of the website. Responsd in markdown format with heading and bullet points.
"""

user_prompt_prefix = """
Here are the contents of a website:
Keep only the text and remove the emojis and other non-text content.
{contents}
"""

rahyab = Website("https://rahyabmohajer.com")

response: Response = openai.responses.create(model="gpt-6-luna", reasoning={"effort": "none"}, input=messages_for(rahyab))

print(response.output_text)



# درباره وب‌سایت رهیاب مهاجر

- **رهیاب مهاجر** خود را «باشگاه ایرانیان مهاجر» معرفی می‌کند و پلتفرمی برای کمک به ایرانیان سراسر جهان است.
- هدف آن گردآوردن خدمات و اطلاعات کاربردی زندگی در مهاجرت در یک جاست؛ از کسب‌وکار و کار گرفته تا رویداد، تخفیف، ملک و نیازمندی‌ها.
- وب‌سایت به زبان فارسی ارائه شده و گزینه‌ای برای تغییر زبان به انگلیسی دارد.

## بخش‌های اصلی وب‌سایت

- کسب‌وکارها
- آگهی‌های شغلی
- تخفیف‌ها
- رویدادها
- تبادلات
- بنرها و نیازمندی‌ها
- املاک و بلاگ و اخبار؛ این بخش‌ها با برچسب «به‌زودی» مشخص شده‌اند.
- ورود و ثبت‌نام

## جستجو و آمار

- صفحه اصلی امکان جستجوی کسب‌وکارها را دارد و نمونه‌ای از عبارت جستجو «وکیل ایرانی» است.
- کاربران می‌توانند کسب‌وکارها را بر اساس شهر پیدا کنند.
- آمار نمایش‌داده‌شده شامل ۲ کسب‌وکار ثبت‌شده، ۱۲ دسته‌بندی فعال، ۱۹ شهر و ۱٬۹۸۲ عضو فعال است.

## دسته‌بندی‌های کسب‌وکار

- حقوقی و مهاجرت
- پزشکی و سلامت
- غذا و رستوران
- منزل و ساختمان
- سالن زیبایی
- خانه و املاک
- مالی و بیزینس
- آموزش
- وسایل نقلیه و باربری
- انجمن و کامیونیتی
- خدمات د